In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from lacan import lacan
from lacan.protect import (
    mol_cleaner,
    protect_rejected_bonds,
    protect_bonds_for_idx,
    get_protected_bond_indices,
    score_mol_ignoring_protected_bonds,
)

# ── Scenario 1: simple repair ─────────────────────────────────────────────────
# Warfarin scores 0 because of its alpha-keto group environment.
# mol_cleaner should mutate that region and return a passing molecule.


profile = lacan.load_profile("chembl")
m1 = Chem.MolFromSmiles("FC1C(F)C(F)C(F)C(C(N2C=NC=N2)(O)N/N=C/C3=CC=CC(F)C3F)=C1")
m2 = Chem.MolFromSmiles("FC1=CC=C(C2=CC=C(C3(CNC(OOCN)=C3)N(C)C4=O)C4=C2)C=C1")
m3 = Chem.MolFromSmiles("FCC1=C(OCl)C=C(N=CN=C2NC3=CC(NN=C4)=C4C=C3)C2=C1")
m4 = Chem.MolFromSmiles("CS(S(=C)(C)=O)C1=CC=BB=C1C2=BBC3=CN=C(NP32)NC(C(OC)=C4)=CN=C4[IH]5(S)NCN(CC(N)=O)CN5B")
for mol in [m4,m1,m2,m3]:
    sc_before, info_before = lacan.score_mol(mol, profile)
    print(f'Input:  score={sc_before:.3f}  bad_bonds={info_before["bad_bonds"]}')
    
    cleaned1 = mol_cleaner(mol, profile, t = 0.05, score_threshold=0.05,
                            max_iter=50, lateral_patience=10)
    
    if cleaned1 is not None:
        sc_after, info_after = lacan.score_mol(cleaned1, profile)
        print(f'Output: score={sc_after:.3f}  bad_bonds={info_after["bad_bonds"]}')
        print(f'SMILES: {Chem.MolToSmiles(cleaned1)}')
        display(Draw.MolsToGridImage(
            [mol, cleaned1], molsPerRow=2, subImgSize=(320, 260),
            legends=[f'Input  score={sc_before:.3f}', f'Cleaned  score={sc_after:.3f}'],
        ))
    else:
        print('mol_cleaner could not find a clean variant — try increasing max_iter.')
    
